

# English Premier League Match Predictor

A machine learning project exploring whether historical match data and recent team performance can be used to predict English Premier League match winners.

## Project Objective

The objective of this project is to develop and evaluate a machine learning model that predicts whether an EPL team will win a match based on information available before the match takes place.

The project focuses on chronological modelling, feature engineering, recent team form, and precision-based evaluation.


In [1]:
# Clone the GitHub repository
!git clone https://github.com/ClPRlAN/EPL-Match-Predictor.git /content/EPL-Match-Predictor

# Import libraries
from pathlib import Path
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
import matplotlib.pyplot as plt
import seaborn as sns

# Define project paths
project_root = Path('/content/EPL-Match-Predictor')
data_path = project_root / 'data' / 'matches.csv'

# Verify the data file exists
if not data_path.exists():
    raise FileNotFoundError(f"Dataset not found at {data_path}")

# Load the dataset
matches = pd.read_csv(data_path, index_col=0)

print("✓ Dataset loaded successfully")
print(f"  Shape: {matches.shape}")
print(f"  Columns: {matches.columns.tolist()}")

Cloning into '/content/EPL-Match-Predictor'...
remote: Enumerating objects: 23, done.
remote: Counting objects: 100% (23/23), done.
remote: Compressing objects: 100% (21/21), done.
remote: Total 23 (delta 6), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (23/23), 69.61 KiB | 1.09 MiB/s, done.
Resolving deltas: 100% (6/6), done.
✓ Dataset loaded successfully
  Shape: (1389, 27)
  Columns: ['date', 'time', 'comp', 'round', 'day', 'venue', 'result', 'gf', 'ga', 'opponent', 'xg', 'xga', 'poss', 'attendance', 'captain', 'formation', 'referee', 'match report', 'notes', 'sh', 'sot', 'dist', 'fk', 'pk', 'pkatt', 'season', 'team']


In [3]:
# Display first few rows
print("First 5 rows of the dataset:")
print(matches.head())
print("\n" + "="*80 + "\n")

First 5 rows of the dataset:
         date   time            comp        round  day venue result   gf   ga  \
1  2021-08-15  16:30  Premier League  Matchweek 1  Sun  Away      L  0.0  1.0   
2  2021-08-21  15:00  Premier League  Matchweek 2  Sat  Home      W  5.0  0.0   
3  2021-08-28  12:30  Premier League  Matchweek 3  Sat  Home      W  5.0  0.0   
4  2021-09-11  15:00  Premier League  Matchweek 4  Sat  Away      W  1.0  0.0   
6  2021-09-18  15:00  Premier League  Matchweek 5  Sat  Home      D  0.0  0.0   

         opponent  ...  match report  notes    sh   sot  dist   fk   pk pkatt  \
1       Tottenham  ...  Match Report    NaN  18.0   4.0  16.9  1.0  0.0   0.0   
2    Norwich City  ...  Match Report    NaN  16.0   4.0  17.3  1.0  0.0   0.0   
3         Arsenal  ...  Match Report    NaN  25.0  10.0  14.3  0.0  0.0   0.0   
4  Leicester City  ...  Match Report    NaN  25.0   8.0  14.0  0.0  0.0   0.0   
6     Southampton  ...  Match Report    NaN  16.0   1.0  15.7  1.0  0.0   0.0  

In [4]:
# Display shape and column information
print("Dataset shape:", matches.shape)
print("\nColumn names:")
print(matches.columns.tolist())
print("\n" + "="*80 + "\n")

Dataset shape: (1389, 27)

Column names:
['date', 'time', 'comp', 'round', 'day', 'venue', 'result', 'gf', 'ga', 'opponent', 'xg', 'xga', 'poss', 'attendance', 'captain', 'formation', 'referee', 'match report', 'notes', 'sh', 'sot', 'dist', 'fk', 'pk', 'pkatt', 'season', 'team']




In [5]:
# Display data types and non-null counts
print("Data types and non-null counts:")
matches.info()
print("\n" + "="*80 + "\n")

Data types and non-null counts:
<class 'pandas.core.frame.DataFrame'>
Index: 1389 entries, 1 to 42
Data columns (total 27 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   date          1389 non-null   object 
 1   time          1389 non-null   object 
 2   comp          1389 non-null   object 
 3   round         1389 non-null   object 
 4   day           1389 non-null   object 
 5   venue         1389 non-null   object 
 6   result        1389 non-null   object 
 7   gf            1389 non-null   float64
 8   ga            1389 non-null   float64
 9   opponent      1389 non-null   object 
 10  xg            1389 non-null   float64
 11  xga           1389 non-null   float64
 12  poss          1389 non-null   float64
 13  attendance    693 non-null    float64
 14  captain       1389 non-null   object 
 15  formation     1389 non-null   object 
 16  referee       1389 non-null   object 
 17  match report  1389 non-null   object 
 18  not

In [6]:
# Display missing values (only columns with missing data)
missing_data = matches.isnull().sum()
missing_data = missing_data[missing_data > 0]

if len(missing_data) > 0:
    print("Missing values by column:")
    print(missing_data)
    print("\nMissing value percentages:")
    print((missing_data / len(matches) * 100).round(2))
else:
    print("No missing values found in the dataset.")

print("\n" + "="*80 + "\n")

Missing values by column:
attendance     696
notes         1389
dist             1
dtype: int64

Missing value percentages:
attendance     50.11
notes         100.00
dist            0.07
dtype: float64




In [7]:
# Display descriptive statistics for numerical columns
print("Descriptive statistics for numerical columns:")
print(matches.describe())
print("\n" + "="*80 + "\n")

Descriptive statistics for numerical columns:
                gf           ga           xg          xga         poss  \
count  1389.000000  1389.000000  1389.000000  1389.000000  1389.000000   
mean      1.335493     1.380850     1.304176     1.338445    49.702664   
std       1.274235     1.291049     0.767268     0.789360    12.401897   
min       0.000000     0.000000     0.000000     0.000000    18.000000   
25%       0.000000     0.000000     0.700000     0.700000    40.000000   
50%       1.000000     1.000000     1.200000     1.200000    50.000000   
75%       2.000000     2.000000     1.800000     1.800000    59.000000   
max       9.000000     9.000000     4.600000     5.000000    82.000000   

         attendance  notes           sh          sot         dist  \
count    693.000000    0.0  1389.000000  1389.000000  1388.000000   
mean   36089.963925    NaN    12.153348     4.041037    17.011527   
std    17797.991778    NaN     5.268876     2.403866     2.988364   
min     200

In [8]:
# Examine the date column
print("Date column inspection:")
print(f"Data type: {matches['date'].dtype}")
print(f"Minimum date: {matches['date'].min()}")
print(f"Maximum date: {matches['date'].max()}")
print("\n" + "="*80 + "\n")

Date column inspection:
Data type: object
Minimum date: 2020-09-12
Maximum date: 2022-04-25




In [9]:
# Identify unique seasons
print("Unique seasons in dataset:")
seasons = sorted(matches['season'].unique())
print(seasons)
print(f"Number of seasons: {len(seasons)}")
print("\n" + "="*80 + "\n")

Unique seasons in dataset:
[np.int64(2021), np.int64(2022)]
Number of seasons: 2




In [10]:
# Identify teams and team counts
print("Number of unique teams:", matches['team'].nunique())
print("\nTeam names (alphabetically sorted):")
teams = sorted(matches['team'].unique())
for team in teams:
    print(f"  {team}")
print("\n" + "="*80 + "\n")

Number of unique teams: 23

Team names (alphabetically sorted):
  Arsenal
  Aston Villa
  Brentford
  Brighton and Hove Albion
  Burnley
  Chelsea
  Crystal Palace
  Everton
  Fulham
  Leeds United
  Leicester City
  Liverpool
  Manchester City
  Manchester United
  Newcastle United
  Norwich City
  Sheffield United
  Southampton
  Tottenham Hotspur
  Watford
  West Bromwich Albion
  West Ham United
  Wolverhampton Wanderers




In [11]:
# Observations per team
print("Number of observations per team:")
team_counts = matches['team'].value_counts().sort_index()
print(team_counts)
print("\n" + "="*80 + "\n")

Number of observations per team:
team
Arsenal                     71
Aston Villa                 70
Brentford                   34
Brighton and Hove Albion    72
Burnley                     71
Chelsea                     70
Crystal Palace              71
Everton                     70
Fulham                      38
Leeds United                71
Leicester City              70
Liverpool                   38
Manchester City             71
Manchester United           72
Newcastle United            72
Norwich City                33
Sheffield United            38
Southampton                 72
Tottenham Hotspur           71
Watford                     33
West Bromwich Albion        38
West Ham United             72
Wolverhampton Wanderers     71
Name: count, dtype: int64




In [12]:
# Observations per season
print("Number of observations per season:")
season_counts = matches['season'].value_counts().sort_index()
print(season_counts)
print("\n" + "="*80 + "\n")

Number of observations per season:
season
2021    760
2022    629
Name: count, dtype: int64




In [13]:
# Distribution of match results
print("Distribution of match results:")
result_dist = matches['result'].value_counts().sort_index()
print(result_dist)
print("\nResult distribution (percentage):")
print((result_dist / len(matches) * 100).round(2))
print("\n" + "="*80 + "\n")

Distribution of match results:
result
D    315
L    548
W    526
Name: count, dtype: int64

Result distribution (percentage):
result
D    22.68
L    39.45
W    37.87
Name: count, dtype: float64




In [14]:
# Expected vs Actual Observations

expected_per_season = 20 * 38
expected_two_seasons = expected_per_season * 2

actual_observations = len(matches)
missing_observations = expected_two_seasons - actual_observations

print(f"Expected observations per complete season: {expected_per_season}")
print(f"Expected observations for two complete seasons: {expected_two_seasons}")
print(f"Actual observations in dataset: {actual_observations}")
print(f"Missing observations from theoretical total: {missing_observations}")

Expected observations per complete season: 760
Expected observations for two complete seasons: 1520
Actual observations in dataset: 1389
Missing observations from theoretical total: 131


In [15]:
# Temporal Coverage

print(f"Earliest date in dataset: {matches['date'].min()}")
print(f"Latest date in dataset: {matches['date'].max()}")
print("\nLatest date by season:")
print(matches.groupby('season')['date'].max())

Earliest date in dataset: 2020-09-12
Latest date in dataset: 2022-04-25

Latest date by season:
season
2021    2021-05-23
2022    2022-04-25
Name: date, dtype: object


In [16]:
# Team Coverage by Season

print("Observations by team and season:")
print(matches.groupby(['team', 'season']).size().unstack(fill_value=0))
print("\nTotal observations per team:")
print(matches.groupby('team').size().sort_values(ascending=False))

Observations by team and season:
season                    2021  2022
team                                
Arsenal                     38    33
Aston Villa                 38    32
Brentford                    0    34
Brighton and Hove Albion    38    34
Burnley                     38    33
Chelsea                     38    32
Crystal Palace              38    33
Everton                     38    32
Fulham                      38     0
Leeds United                38    33
Leicester City              38    32
Liverpool                   38     0
Manchester City             38    33
Manchester United           38    34
Newcastle United            38    34
Norwich City                 0    33
Sheffield United            38     0
Southampton                 38    34
Tottenham Hotspur           38    33
Watford                      0    33
West Bromwich Albion        38     0
West Ham United             38    34
Wolverhampton Wanderers     38    33

Total observations per team:
team
Brighto

In [17]:
# Coverage Anomalies

teams_by_season = matches.groupby(['team', 'season']).size().unstack(fill_value=0)

teams_one_season = teams_by_season[(teams_by_season[2021] == 0) | (teams_by_season[2022] == 0)].index.tolist()
print(f"Teams with data for only one season: {teams_one_season}")

season_2022_by_team = matches[matches['season'] == 2022].groupby('team').size()
incomplete_2022 = season_2022_by_team[season_2022_by_team < 38].sort_values(ascending=False)
print(f"\nTeams with fewer than 38 observations in 2022 season:")
print(incomplete_2022)

Teams with data for only one season: ['Brentford', 'Fulham', 'Liverpool', 'Norwich City', 'Sheffield United', 'Watford', 'West Bromwich Albion']

Teams with fewer than 38 observations in 2022 season:
team
Brentford                   34
Southampton                 34
Brighton and Hove Albion    34
West Ham United             34
Manchester United           34
Newcastle United            34
Arsenal                     33
Norwich City                33
Burnley                     33
Crystal Palace              33
Leeds United                33
Wolverhampton Wanderers     33
Tottenham Hotspur           33
Watford                     33
Manchester City             33
Aston Villa                 32
Leicester City              32
Chelsea                     32
Everton                     32
dtype: int64


In [18]:
# Partial Season Missing Observations

obs_2022 = len(matches[matches['season'] == 2022])
expected_complete = 20 * 38

print(f"Actual observations in season 2022: {obs_2022}")
print(f"Expected observations in complete season: {expected_complete}")
print(f"Missing observations in season 2022: {expected_complete - obs_2022}")

Actual observations in season 2022: 629
Expected observations in complete season: 760
Missing observations in season 2022: 131


### Key Data Quality Findings

The dataset contains 1,389 observations across two seasons, which is 131 short of the theoretical 1,520 (760 per complete season). This shortfall is entirely attributable to the incomplete 2022 season, which was scraped through April 25, 2022. The 2021 season is complete with 760 team-level observations. Teams naturally have fewer observations in 2022—between 32 and 34 for continuing Premier League teams—because the season had not concluded at the time of data collection.

Promotion and relegation explain why certain teams appear in only one season. Fulham, Sheffield United, and West Bromwich Albion have 38 observations each from the 2021 season because they were relegated before the 2022 season. Brentford, Norwich City, and Watford appear only in 2022 following promotion. Liverpool is the notable data-quality anomaly: unlike the other teams, it should have observations in both seasons but contains data only from 2021, indicating a scraping issue. Rather than impute or artificially reconstruct missing matches, I will retain the dataset as-is and treat this limitation explicitly when interpreting the model's results.

## Data Cleaning and Preparation

Before modelling, I need to prepare the raw dataset for analysis while preserving information that could be useful for feature engineering. This involves inspecting data types, converting temporal variables to appropriate formats, removing clearly non-informative columns, and verifying the structure of the cleaned dataset.

In [19]:
# Inspect Predictor Data Types

print("Data Types of Potential Predictor Columns")
print("=" * 50)

predictor_cols = ['venue', 'opponent', 'time', 'day', 'gf', 'ga', 'xg', 'xga',
                  'poss', 'sh', 'sot', 'dist', 'fk', 'pk', 'pkatt']

for col in predictor_cols:
    if col in matches.columns:
        dtype = matches[col].dtype
        n_unique = matches[col].nunique()
        print(f"{col:12} | dtype: {str(dtype):10} | unique values: {n_unique}")
    else:
        print(f"{col:12} | NOT FOUND")

Data Types of Potential Predictor Columns
venue        | dtype: object     | unique values: 2
opponent     | dtype: object     | unique values: 23
time         | dtype: object     | unique values: 18
day          | dtype: object     | unique values: 7
gf           | dtype: float64    | unique values: 9
ga           | dtype: float64    | unique values: 9
xg           | dtype: float64    | unique values: 44
xga          | dtype: float64    | unique values: 45
poss         | dtype: float64    | unique values: 64
sh           | dtype: float64    | unique values: 32
sot          | dtype: float64    | unique values: 16
dist         | dtype: float64    | unique values: 160
fk           | dtype: float64    | unique values: 5
pk           | dtype: float64    | unique values: 4
pkatt        | dtype: float64    | unique values: 4


In [20]:
# Convert Date to Datetime

matches['date'] = pd.to_datetime(matches['date'])

print("Date Conversion Complete")
print("=" * 50)
print(f"Date dtype: {matches['date'].dtype}")
print(f"Earliest match: {matches['date'].min()}")
print(f"Latest match: {matches['date'].max()}")

Date Conversion Complete
Date dtype: datetime64[ns]
Earliest match: 2020-09-12 00:00:00
Latest match: 2022-04-25 00:00:00


In [21]:
# Remove Non-Informative Columns

# Columns to exclude from modelling dataset:
# - comp: all values are "Premier League" (constant)
# - round: redundant with date information
# - match report: URL/link, not predictive
# - notes: sparse, game-specific annotations
# - attendance: stadium capacity not a team performance indicator
# - captain: player names, no aggregated predictive signal
# - formation: could be useful but requires careful encoding; defer to future work
# - referee: individual referee effect is marginal for this analysis

cols_to_drop = ['comp', 'round', 'match report', 'notes', 'attendance',
                'captain', 'formation', 'referee']

matches_clean = matches.drop(columns=cols_to_drop)

print(f"Original dataset shape: {matches.shape}")
print(f"Cleaned dataset shape: {matches_clean.shape}")
print(f"Columns removed: {len(cols_to_drop)}")

Original dataset shape: (1389, 27)
Cleaned dataset shape: (1389, 19)
Columns removed: 8


In [22]:
# Verify Clean Dataset

print("Cleaned Dataset Structure")
print("=" * 50)
print(f"\nShape: {matches_clean.shape}")
print(f"\nColumns:\n{list(matches_clean.columns)}")
print(f"\nData Types:\n{matches_clean.dtypes}")
print(f"\nMissing Values:\n{matches_clean.isnull().sum()}")

Cleaned Dataset Structure

Shape: (1389, 19)

Columns:
['date', 'time', 'day', 'venue', 'result', 'gf', 'ga', 'opponent', 'xg', 'xga', 'poss', 'sh', 'sot', 'dist', 'fk', 'pk', 'pkatt', 'season', 'team']

Data Types:
date        datetime64[ns]
time                object
day                 object
venue               object
result              object
gf                 float64
ga                 float64
opponent            object
xg                 float64
xga                float64
poss               float64
sh                 float64
sot                float64
dist               float64
fk                 float64
pk                 float64
pkatt              float64
season               int64
team                object
dtype: object

Missing Values:
date        0
time        0
day         0
venue       0
result      0
gf          0
ga          0
opponent    0
xg          0
xga         0
poss        0
sh          0
sot         0
dist        1
fk          0
pk          0
pkatt       0
se

## Target and Baseline Feature Engineering

Before training a model, I need to define the prediction target and create a set of baseline features that capture match context without using post-match statistics.

The target will be binary: a team wins (`1`) or does not win (`0`). This simplification treats draws and losses equivalently, reflecting the perspective that from a predictive standpoint, only wins are categorically different.

Baseline features will encode match venue, opponent identity, and temporal characteristics. These features are derived from information available before a match is played and do not depend on the match outcome.

In [23]:
# Create binary target: Win = 1, Draw/Loss = 0
matches_clean['target'] = (matches_clean['result'] == 'W').astype(int)

# Display target distribution
print("Target Value Counts:")
print(matches_clean['target'].value_counts())
print("\nTarget Distribution (%):")
print(matches_clean['target'].value_counts(normalize=True) * 100)

Target Value Counts:
target
0    863
1    526
Name: count, dtype: int64

Target Distribution (%):
target
0    62.13103
1    37.86897
Name: proportion, dtype: float64


In [24]:
# Encode venue and opponent as numeric codes
matches_clean['venue_code'] = matches_clean['venue'].astype('category').cat.codes
matches_clean['opp_code'] = matches_clean['opponent'].astype('category').cat.codes

# Display mapping and sample
print("Venue Encoding Sample:")
venue_mapping = matches_clean[['venue', 'venue_code']].drop_duplicates().sort_values('venue_code')
print(venue_mapping)

print("\n" + "="*60)
print("Opponent Encoding Sample:")
opp_mapping = matches_clean[['opponent', 'opp_code']].drop_duplicates().sort_values('opp_code').head(10)
print(opp_mapping)

print("\n" + "="*60)
print("Sample of Original and Encoded Columns:")
print(matches_clean[['venue', 'venue_code', 'opponent', 'opp_code']].head(8))

Venue Encoding Sample:
  venue  venue_code
1  Away           0
2  Home           1

Opponent Encoding Sample:
          opponent  opp_code
3          Arsenal         0
21     Aston Villa         1
28       Brentford         2
13        Brighton         3
11         Burnley         4
8          Chelsea         5
15  Crystal Palace         6
18         Everton         7
16          Fulham         8
25    Leeds United         9

Sample of Original and Encoded Columns:
   venue  venue_code        opponent  opp_code
1   Away           0       Tottenham        18
2   Home           1    Norwich City        15
3   Home           1         Arsenal         0
4   Away           0  Leicester City        10
6   Home           1     Southampton        17
8   Away           0         Chelsea         5
10  Away           0       Liverpool        11
11  Home           1         Burnley         4


In [25]:
# Extract hour from time column (format: "HH:MM")
matches_clean['hour'] = matches_clean['time'].str.split(':').str[0].astype(float)

# Extract day of week (0=Monday, 6=Sunday)
matches_clean['day_code'] = matches_clean['date'].dt.dayofweek

# Display sample
print("Temporal Predictors Sample:")
print(matches_clean[['date', 'time', 'hour', 'day_code']].head(10))

print("\nDay of Week Distribution:")
day_names = {0: 'Monday', 1: 'Tuesday', 2: 'Wednesday', 3: 'Thursday',
             4: 'Friday', 5: 'Saturday', 6: 'Sunday'}
print(matches_clean['day_code'].map(day_names).value_counts().sort_index())

Temporal Predictors Sample:
         date   time  hour  day_code
1  2021-08-15  16:30  16.0         6
2  2021-08-21  15:00  15.0         5
3  2021-08-28  12:30  12.0         5
4  2021-09-11  15:00  15.0         5
6  2021-09-18  15:00  15.0         5
8  2021-09-25  12:30  12.0         5
10 2021-10-03  16:30  16.0         6
11 2021-10-16  15:00  15.0         5
13 2021-10-23  17:30  17.0         5
15 2021-10-30  15:00  15.0         5

Day of Week Distribution:
day_code
Friday        54
Monday       104
Saturday     551
Sunday       417
Thursday      50
Tuesday       88
Wednesday    125
Name: count, dtype: int64


In [26]:
# Define baseline predictor columns
predictors = ["venue_code", "opp_code", "hour", "day_code"]

# Verify data types
print("Predictor Data Types:")
print(matches_clean[predictors].dtypes)

print("\n" + "="*60)
print("Missing Values in Predictors:")
print(matches_clean[predictors].isnull().sum())

print("\n" + "="*60)
print("Sample of Predictors with Target:")
print(matches_clean[predictors + ['target']].head(10))

Predictor Data Types:
venue_code       int8
opp_code         int8
hour          float64
day_code        int32
dtype: object

Missing Values in Predictors:
venue_code    0
opp_code      0
hour          0
day_code      0
dtype: int64

Sample of Predictors with Target:
    venue_code  opp_code  hour  day_code  target
1            0        18  16.0         6       0
2            1        15  15.0         5       1
3            1         0  12.0         5       1
4            0        10  15.0         5       1
6            1        17  15.0         5       0
8            0         5  12.0         5       1
10           0        11  16.0         6       0
11           1         4  15.0         5       1
13           0         3  17.0         5       1
15           1         6  15.0         5       0


In [27]:
# Chronological train/test split: pre-2022 training, 2022 test
train = matches_clean[matches_clean['season'] < 2022].copy()
test = matches_clean[matches_clean['season'] == 2022].copy()

X_train = train[predictors]
y_train = train['target']

X_test = test[predictors]
y_test = test['target']

print(f"Training set: {len(train)} observations")
print(f"Training date range: {train['date'].min()} to {train['date'].max()}")
print(f"\nTest set: {len(test)} observations")
print(f"Test date range: {test['date'].min()} to {test['date'].max()}")
print(f"\nFeatures used: {predictors}")

Training set: 760 observations
Training date range: 2020-09-12 00:00:00 to 2021-05-23 00:00:00

Test set: 629 observations
Test date range: 2021-08-13 00:00:00 to 2022-04-25 00:00:00

Features used: ['venue_code', 'opp_code', 'hour', 'day_code']


In [28]:
from sklearn.ensemble import RandomForestClassifier

rf_baseline = RandomForestClassifier(
    n_estimators=100,
    min_samples_split=10,
    random_state=1,
    n_jobs=-1
)

rf_baseline.fit(X_train, y_train)

print("Baseline Random Forest trained.")
print(f"Training set size: {len(X_train)}")
print(f"Number of features: {X_train.shape[1]}")

Baseline Random Forest trained.
Training set size: 760
Number of features: 4


In [29]:
preds = rf_baseline.predict(X_test)

print(f"Total predictions: {len(preds)}")
print(f"Predicted wins: {preds.sum()}")
print(f"Predicted non-wins: {(preds == 0).sum()}")
print(f"\nFirst 10 predictions: {preds[:10]}")

Total predictions: 629
Predicted wins: 110
Predicted non-wins: 519

First 10 predictions: [0 0 0 0 1 0 0 1 0 0]


In [30]:
from sklearn.metrics import precision_score, confusion_matrix

precision_baseline = precision_score(y_test, preds)

print(f"Baseline Precision: {precision_baseline:.1%}")
print(f"\nConfusion Matrix:")
print(confusion_matrix(y_test, preds))
print(f"\nTrue Positives: {((preds == 1) & (y_test == 1)).sum()}")
print(f"False Positives: {((preds == 1) & (y_test == 0)).sum()}")
print(f"True Negatives: {((preds == 0) & (y_test == 0)).sum()}")
print(f"False Negatives: {((preds == 0) & (y_test == 1)).sum()}")

Baseline Precision: 45.5%

Confusion Matrix:
[[340  60]
 [179  50]]

True Positives: 50
False Positives: 60
True Negatives: 340
False Negatives: 179


## Recent-Form Feature Engineering

I will now engineer rolling historical features that capture each team's recent performance. By shifting the data before calculating rolling averages, I ensure that each match uses only information available before that match, preventing data leakage.



In [31]:
matches_rolling = matches_clean.copy()
matches_rolling = matches_rolling.sort_values(['team', 'date']).reset_index(drop=True)

In [32]:
rolling_columns = ['gf', 'ga', 'sh', 'sot', 'dist', 'fk', 'pk', 'pkatt']

for col in rolling_columns:
    matches_rolling[f'{col}_rolling'] = (
        matches_rolling.groupby('team')[col]
        .shift(1)
        .rolling(window=3, min_periods=1)
        .mean()
        .reset_index(level=0, drop=True)
    )

In [33]:
display(matches_rolling[['team', 'date', 'gf', 'gf_rolling', 'ga', 'ga_rolling', 'sh', 'sh_rolling']].head(15))
print("\nMissing values in rolling features:")
print(matches_rolling[['gf_rolling', 'ga_rolling', 'sh_rolling', 'sot_rolling', 'dist_rolling', 'fk_rolling', 'pk_rolling', 'pkatt_rolling']].isnull().sum())

,team,date,gf,gf_rolling,ga,ga_rolling,sh,sh_rolling
0,Arsenal,2020-09-12,3.0,NaN,0.0,NaN,13.0,NaN
1,Arsenal,2020-09-19,2.0,3.000000,1.0,0.000000,6.0,13.000000
2,Arsenal,2020-09-28,1.0,2.500000,3.0,0.500000,4.0,9.500000
3,Arsenal,2020-10-04,2.0,2.000000,1.0,1.333333,6.0,7.666667
4,Arsenal,2020-10-17,0.0,1.666667,1.0,1.666667,11.0,5.333333
5,Arsenal,2020-10-25,0.0,1.000000,1.0,1.666667,12.0,7.000000
6,Arsenal,2020-11-01,1.0,0.666667,0.0,1.000000,6.0,9.666667
7,Arsenal,2020-11-08,0.0,0.333333,3.0,0.666667,13.0,9.666667
8,Arsenal,2020-11-22,0.0,0.333333,0.0,1.333333,9.0,10.333333
9,Arsenal,2020-11-29,1.0,0.333333,2.0,1.000000,13.0,9.333333



Missing values in rolling features:
gf_rolling       1
ga_rolling       1
sh_rolling       1
sot_rolling      1
dist_rolling     1
fk_rolling       1
pk_rolling       1
pkatt_rolling    1
dtype: int64


In [34]:
sample_team = matches_rolling[matches_rolling['team'] == 'Manchester City'].sort_values('date').head(10)
display(sample_team[['date', 'gf', 'gf_rolling']])

,date,gf,gf_rolling
746,2020-09-21,3.0,2.500000
747,2020-09-27,2.0,3.000000
748,2020-10-03,1.0,2.500000
749,2020-10-17,1.0,2.000000
750,2020-10-24,1.0,1.333333
751,2020-10-31,1.0,1.000000
752,2020-11-08,1.0,1.000000
753,2020-11-21,0.0,1.000000
754,2020-11-28,5.0,0.666667
755,2020-12-05,2.0,2.000000


In [35]:
rolling_predictors = [
    'venue_code', 'opp_code', 'hour', 'day_code',
    'gf_rolling', 'ga_rolling', 'sh_rolling', 'sot_rolling',
    'dist_rolling', 'fk_rolling', 'pk_rolling', 'pkatt_rolling'
]

# Verify all columns exist
missing_cols = [col for col in rolling_predictors if col not in matches_rolling.columns]
if missing_cols:
    print(f"Missing columns: {missing_cols}")
else:
    print(f"✓ All {len(rolling_predictors)} rolling predictors present in matches_rolling")

✓ All 12 rolling predictors present in matches_rolling


In [36]:
matches_model = matches_rolling.copy()

# Record original size
original_rows = len(matches_model)

# Drop rows with missing values in rolling predictor columns
matches_model = matches_model.dropna(subset=rolling_predictors)

# Calculate removal statistics
rows_removed = original_rows - len(matches_model)
final_rows = len(matches_model)

print(f"Original rows:        {original_rows}")
print(f"Rows removed:         {rows_removed}")
print(f"Final rows:           {final_rows}")
print(f"Removal rate:         {rows_removed/original_rows*100:.1f}%")
print(f"\nRemaining NaN values in predictors: {matches_model[rolling_predictors].isna().sum().sum()}")

Original rows:        1389
Rows removed:         1
Final rows:           1388
Removal rate:         0.1%

Remaining NaN values in predictors: 0


In [38]:
# Create chronological train/test split for rolling features
train_rolling = matches_model[matches_model['season'] < 2022].copy()
test_rolling = matches_model[matches_model['season'] == 2022].copy()

# Prepare feature and target arrays
X_train_rolling = train_rolling[rolling_predictors]
y_train_rolling = train_rolling['target']
X_test_rolling = test_rolling[rolling_predictors]
y_test_rolling = test_rolling['target']

print("Training Set (season < 2022):")
print(f"  Observations: {len(train_rolling)}")
print(f"  Date range: {train_rolling['date'].min()} to {train_rolling['date'].max()}")
print(f"  Features shape: {X_train_rolling.shape}")
print(f"  Target shape: {y_train_rolling.shape}")

print("\nTest Set (season == 2022):")
print(f"  Observations: {len(test_rolling)}")
print(f"  Date range: {test_rolling['date'].min()} to {test_rolling['date'].max()}")
print(f"  Features shape: {X_test_rolling.shape}")
print(f"  Target shape: {y_test_rolling.shape}")

print(f"\nTraining/Test ratio: {len(train_rolling)/(len(train_rolling)+len(test_rolling))*100:.1f}% / {len(test_rolling)/(len(train_rolling)+len(test_rolling))*100:.1f}%")

Training Set (season < 2022):
  Observations: 759
  Date range: 2020-09-12 00:00:00 to 2021-05-23 00:00:00
  Features shape: (759, 12)
  Target shape: (759,)

Test Set (season == 2022):
  Observations: 629
  Date range: 2021-08-13 00:00:00 to 2022-04-25 00:00:00
  Features shape: (629, 12)
  Target shape: (629,)

Training/Test ratio: 54.7% / 45.3%


In [39]:
from sklearn.ensemble import RandomForestClassifier

# Initialize and train the improved model with rolling features
rf_rolling = RandomForestClassifier(
    n_estimators=100,
    min_samples_split=10,
    random_state=1
)

rf_rolling.fit(X_train_rolling, y_train_rolling)

print("Improved Random Forest Model trained successfully")
print(f"Training observations: {X_train_rolling.shape[0]}")
print(f"Number of predictors: {X_train_rolling.shape[1]}")

Improved Random Forest Model trained successfully
Training observations: 759
Number of predictors: 12


In [40]:
# Generate predictions on test set
preds_rolling = rf_rolling.predict(X_test_rolling)

print(f"Total predictions: {len(preds_rolling)}")
print(f"Predicted wins (1): {(preds_rolling == 1).sum()}")
print(f"Predicted non-wins (0): {(preds_rolling == 0).sum()}")
print(f"\nFirst 10 predictions: {preds_rolling[:10]}")

Total predictions: 629
Predicted wins (1): 115
Predicted non-wins (0): 514

First 10 predictions: [0 1 0 1 0 0 1 0 0 0]


In [41]:
from sklearn.metrics import precision_score, confusion_matrix

# Calculate precision
precision_rolling = precision_score(y_test_rolling, preds_rolling)

# Confusion matrix
tn, fp, fn, tp = confusion_matrix(y_test_rolling, preds_rolling).ravel()

print(f"Improved Model Precision: {precision_rolling * 100:.1f}%")
print(f"\nConfusion Matrix:")
print(f"  True Positives (correct wins): {tp}")
print(f"  False Positives (incorrect wins): {fp}")
print(f"  True Negatives (correct non-wins): {tn}")
print(f"  False Negatives (missed wins): {fn}")

print(f"\n--- Model Comparison ---")
print(f"Baseline Model Precision: 45.5%")
print(f"Improved Model Precision: {precision_rolling * 100:.1f}%")
print(f"Absolute Improvement: {(precision_rolling * 100) - 45.5:.1f} percentage points")

Improved Model Precision: 53.0%

Confusion Matrix:
  True Positives (correct wins): 61
  False Positives (incorrect wins): 54
  True Negatives (correct non-wins): 346
  False Negatives (missed wins): 168

--- Model Comparison ---
Baseline Model Precision: 45.5%
Improved Model Precision: 53.0%
Absolute Improvement: 7.5 percentage points


## Match-Level Prediction Analysis

Each Premier League fixture appears twice in the dataset, once from the perspective of each team. I therefore aligned the two team-level predictions for the same fixture to identify cases where the model predicts exactly one team to win.

This additional filtering removes ambiguous predictions, such as cases where both teams are predicted to win, and allows me to evaluate whether restricting predictions to clear winner-loser scenarios improves precision.

In [42]:
predictions = test_rolling[['date', 'team', 'opponent', 'target']].copy()
predictions['predicted'] = preds_rolling

print(predictions.head(10))

         date     team         opponent  target  predicted
38 2021-08-13  Arsenal        Brentford       0          0
39 2021-08-22  Arsenal          Chelsea       0          1
40 2021-08-28  Arsenal  Manchester City       0          0
41 2021-09-11  Arsenal     Norwich City       1          1
42 2021-09-18  Arsenal          Burnley       1          0
43 2021-09-26  Arsenal        Tottenham       1          0
44 2021-10-02  Arsenal         Brighton       0          1
45 2021-10-18  Arsenal   Crystal Palace       0          0
46 2021-10-22  Arsenal      Aston Villa       1          0
47 2021-10-30  Arsenal   Leicester City       1          0


In [43]:
map_values = {
    "Brighton and Hove Albion": "Brighton",
    "Manchester United": "Manchester Utd",
    "Newcastle United": "Newcastle Utd",
    "Tottenham Hotspur": "Tottenham",
    "West Ham United": "West Ham",
    "Wolverhampton Wanderers": "Wolves"
}

predictions_aligned = predictions.copy()
predictions_aligned['team'] = predictions_aligned['team'].map(map_values).fillna(predictions_aligned['team'])

print(predictions_aligned.head(10))

         date     team         opponent  target  predicted
38 2021-08-13  Arsenal        Brentford       0          0
39 2021-08-22  Arsenal          Chelsea       0          1
40 2021-08-28  Arsenal  Manchester City       0          0
41 2021-09-11  Arsenal     Norwich City       1          1
42 2021-09-18  Arsenal          Burnley       1          0
43 2021-09-26  Arsenal        Tottenham       1          0
44 2021-10-02  Arsenal         Brighton       0          1
45 2021-10-18  Arsenal   Crystal Palace       0          0
46 2021-10-22  Arsenal      Aston Villa       1          0
47 2021-10-30  Arsenal   Leicester City       1          0


In [44]:
match_predictions = predictions_aligned.merge(
    predictions_aligned,
    left_on=['date', 'team', 'opponent'],
    right_on=['date', 'opponent', 'team'],
    suffixes=('_home', '_away')
)

# Remove duplicates by keeping only one direction of each match
match_predictions = match_predictions[match_predictions['team_home'] < match_predictions['team_away']].reset_index(drop=True)

print(f"Number of matched fixtures: {len(match_predictions)}")
print(f"\nFirst 10 rows:")
print(match_predictions[['date', 'team_home', 'team_away', 'predicted_home', 'predicted_away', 'target_home', 'target_away']].head(10))
print(f"\nColumns: {match_predictions.columns.tolist()}")

Number of matched fixtures: 298

First 10 rows:
        date team_home        team_away  predicted_home  predicted_away  \
0 2021-08-13   Arsenal        Brentford               0               0   
1 2021-08-22   Arsenal          Chelsea               1               0   
2 2021-08-28   Arsenal  Manchester City               0               1   
3 2021-09-11   Arsenal     Norwich City               1               0   
4 2021-09-18   Arsenal          Burnley               0               0   
5 2021-09-26   Arsenal        Tottenham               0               0   
6 2021-10-02   Arsenal         Brighton               1               0   
7 2021-10-18   Arsenal   Crystal Palace               0               0   
8 2021-10-22   Arsenal      Aston Villa               0               0   
9 2021-10-30   Arsenal   Leicester City               0               0   

   target_home  target_away  
0            0            1  
1            0            1  
2            0            1  
3     

In [45]:
# Filter for matches where exactly one team is predicted to win
consistent = match_predictions[
    ((match_predictions['predicted_home'] == 1) & (match_predictions['predicted_away'] == 0)) |
    ((match_predictions['predicted_home'] == 0) & (match_predictions['predicted_away'] == 1))
].reset_index(drop=True)

# Determine which team was predicted to win
predicted_winner = []
actual_winner = []

for idx, row in consistent.iterrows():
    if row['predicted_home'] == 1:
        # Home team predicted to win
        predicted_winner.append(row['target_home'] == 1)
    else:
        # Away team predicted to win
        predicted_winner.append(row['target_away'] == 1)

correct = sum(predicted_winner)
total = len(consistent)
match_level_precision = (correct / total * 100) if total > 0 else 0

print(f"Total matched fixtures: {len(match_predictions)}")
print(f"Fixtures with exactly one predicted winner: {total}")
print(f"Correct predictions: {correct}")
print(f"Incorrect predictions: {total - correct}")
print(f"Match-level precision: {match_level_precision:.1f}%")
print(f"\nComparison:")
print(f"  Team-level improved precision: 53.0%")
print(f"  Match-level precision (filtered): {match_level_precision:.1f}%")

Total matched fixtures: 298
Fixtures with exactly one predicted winner: 105
Correct predictions: 58
Incorrect predictions: 47
Match-level precision: 55.2%

Comparison:
  Team-level improved precision: 53.0%
  Match-level precision (filtered): 55.2%


In [46]:
# Final validation: check for duplicated fixtures

fixture_keys = ['date', 'team_home', 'team_away']

total_rows = len(match_predictions)
unique_fixtures = match_predictions[fixture_keys].drop_duplicates().shape[0]

duplicated_rows = match_predictions.duplicated(
    subset=fixture_keys,
    keep=False
).sum()

duplicated_groups = (
    match_predictions[match_predictions.duplicated(
        subset=fixture_keys,
        keep=False
    )]
    .groupby(fixture_keys)
    .ngroups
)

print("Match-Level Merge Validation")
print("=" * 50)
print(f"Total rows:              {total_rows}")
print(f"Unique fixtures:         {unique_fixtures}")
print(f"Duplicated fixture rows: {duplicated_rows}")
print(f"Duplicated fixture groups: {duplicated_groups}")

Match-Level Merge Validation
Total rows:              298
Unique fixtures:         298
Duplicated fixture rows: 0
Duplicated fixture groups: 0


In [47]:
# Final model performance summary

filtered_predictions = len(
    match_predictions[
        (match_predictions['predicted_home'] != match_predictions['predicted_away'])
    ]
)

filtered_precision = 55.2  # previously calculated from the actual filtered predictions

baseline_precision = 45.5
improved_precision = 53.0

summary = pd.DataFrame({
    'Model': [
        'Baseline Random Forest',
        'Improved Random Forest',
        'Filtered Match-Level Predictions'
    ],
    'Precision (%)': [
        baseline_precision,
        improved_precision,
        filtered_precision
    ],
    'Number of Predictions': [
        len(preds),
        len(preds_rolling),
        filtered_predictions
    ]
})

display(summary)

print("\nPrecision Improvements")
print("=" * 50)
print(f"Improved vs Baseline:       {improved_precision - baseline_precision:.1f} percentage points")
print(f"Filtered vs Improved:       {filtered_precision - improved_precision:.1f} percentage points")

,Model,Precision (%),Number of Predictions
0,Baseline Random Forest,45.5,629
1,Improved Random Forest,53.0,629
2,Filtered Match-Level Predictions,55.2,105



Precision Improvements
Improved vs Baseline:       7.5 percentage points
Filtered vs Improved:       2.2 percentage points


## Final Results

The baseline Random Forest achieved a precision of 45.5% on the 2022 test set, identifying 50 wins correctly from 110 predicted wins.

After introducing rolling recent-form features, the improved Random Forest achieved a precision of 53.0%, representing an absolute improvement of 7.5 percentage points over the baseline.

At match level, predictions were reconstructed by pairing the home and away observations belonging to the same fixture. Only matches where the model predicted exactly one winner were retained. This produced 105 filtered match-level predictions with a precision of 55.2%.

The results indicate that recent-form features improved the model's ability to identify winning outcomes. However, the higher match-level precision comes with substantially lower prediction coverage, since only 105 of the 298 matched fixtures generated a single predicted winner.

## Interpretation

The main improvement comes from incorporating recent team performance rather than relying only on static match context such as venue, opponent, and kick-off time.

The improved model increased precision from 45.5% to 53.0%, suggesting that recent-form information provides useful predictive signal. However, precision alone does not describe the complete performance of the model. The model predicted relatively few wins compared with the total number of test observations, and the filtered match-level approach further reduced coverage.

The match-level analysis is particularly important because each fixture appears from both teams' perspectives in the original dataset. Reconstructing fixtures allows predictions to be evaluated in the context of an actual match rather than as two independent team-level observations.

Therefore, the filtered match-level result of 55.2% should not be interpreted as a universally superior model performance. It represents a higher-precision strategy that makes predictions only when the two team-level predictions produce exactly one predicted winner.

## Limitations

This project has several limitations that should be considered when interpreting the results.

First, the dataset covers only two Premier League seasons, and the 2021–22 season is incomplete because the data was collected in April 2022. In addition, Liverpool is missing from the second season due to a source-data scraping issue. These limitations reduce the amount of historical information available to the model.

Second, the model uses a relatively small set of predictors. The rolling features capture recent team performance, but they do not account for factors such as injuries, player availability, squad changes, managerial changes, or the strength of the opponent's recent form.

Finally, precision was used as the primary evaluation metric because the objective is to understand how often a predicted win is actually a win. The match-level filtering strategy improves precision, but it also substantially reduces the number of predictions made. A higher precision therefore does not necessarily mean that the strategy would generate more profitable or useful predictions in a real-world betting or forecasting environment.



## Future Work

Several extensions could make this project more robust.

- Add additional Premier League seasons to increase the amount of training data.
- Incorporate opponent-specific rolling statistics.
- Engineer additional features from expected goals, possession, shots, and other match statistics.
- Include information about player availability, injuries, and squad changes where reliable historical data is available.
- Compare the Random Forest with alternative classification algorithms.
- Evaluate additional metrics such as recall, F1-score, ROC-AUC, and calibration.
- Test the approach on other football leagues or competitions.
- Develop a more rigorous walk-forward validation framework instead of relying on a single chronological train/test split.

These extensions would allow the model to be evaluated across a wider range of historical conditions and provide a more complete assessment of its predictive performance.